<a href="https://colab.research.google.com/github/alexgohenghong/MRO-Spare-Parts-Inventory-Optimizer/blob/main/mro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# PHASE 1: High-Value Aerospace Component Inventory Simulation
# Objective: Generate synthetic demand and lead time data for Avionics MRO parts
# ==============================================================================
import pandas as pd
import numpy as np
import math
from IPython.display import display

# Simulate 50 high-value avionics parts (e.g., Honeywell HW-AV series)
def generate_mro_data(n_parts=50):
    np.random.seed(42)
    data = []
    for i in range(n_parts):
        data.append({
            'Part_ID': f'HW-AV-{1000+i}',
            'Unit_Cost_RM': round(np.random.uniform(10000, 150000), 2), # Currency: RM
            'Avg_Monthly_Demand': round(np.random.uniform(5, 50)),
            'Demand_Std_Dev': round(np.random.uniform(2, 15)),
            'Lead_Time_Months': round(np.random.uniform(1, 4), 1)
        })
    return pd.DataFrame(data)

mro_df = generate_mro_data()
print("✅ Phase 1 Complete: MRO Inventory Dataset Ready.")
display(mro_df.head())

✅ Phase 1 Complete: MRO Inventory Dataset Ready.


,Part_ID,Unit_Cost_RM,Avg_Monthly_Demand,Demand_Std_Dev,Lead_Time_Months
0,HW-AV-1000,62435.62,48,12,2.8
1,HW-AV-1001,31842.61,12,3,3.6
2,HW-AV-1002,94156.10,37,2,3.9
3,HW-AV-1003,126541.97,15,4,1.6
4,HW-AV-1004,52593.91,29,8,1.9


In [4]:
# ==============================================================================
# PHASE 2: Lean Inventory Optimization (Reorder Point & Safety Stock)
# Objective: Apply statistical modeling to minimize capital tie-up at a 95% service level
# ==============================================================================

def optimize_inventory(df, target_service_level=0.95):
    # Z-Score for 95% Service Level is approx 1.645
    z_score = 1.645

    # Calculate Optimized Safety Stock
    # Formula: Z * sqrt(Lead_Time) * Standard_Deviation
    df['Optimized_Safety_Stock'] = np.ceil(
        z_score * np.sqrt(df['Lead_Time_Months']) * df['Demand_Std_Dev']
    )

    # Calculate Reorder Point (ROP)
    df['Reorder_Point'] = np.ceil(
        (df['Avg_Monthly_Demand'] * df['Lead_Time_Months']) + df['Optimized_Safety_Stock']
    )

    # Compare against Legacy "Rule of Thumb" (e.g., flat 3 months of demand as safety stock)
    df['Legacy_Safety_Stock'] = np.ceil(df['Avg_Monthly_Demand'] * 3)

    df['Legacy_Capital_Tied_RM'] = df['Legacy_Safety_Stock'] * df['Unit_Cost_RM']
    df['Optimized_Capital_Tied_RM'] = df['Optimized_Safety_Stock'] * df['Unit_Cost_RM']

    return df

optimized_df = optimize_inventory(mro_df)
print("✅ Phase 2 Complete: Lean Inventory Thresholds Calculated.")
display(optimized_df[['Part_ID', 'Legacy_Safety_Stock', 'Optimized_Safety_Stock', 'Reorder_Point']].head())

✅ Phase 2 Complete: Lean Inventory Thresholds Calculated.


,Part_ID,Legacy_Safety_Stock,Optimized_Safety_Stock,Reorder_Point
0,HW-AV-1000,144.0,34.0,169.0
1,HW-AV-1001,36.0,10.0,54.0
2,HW-AV-1002,111.0,7.0,152.0
3,HW-AV-1003,45.0,9.0,33.0
4,HW-AV-1004,87.0,19.0,75.0


In [5]:
# ==============================================================================
# PHASE 3: Financial ROI & AOG Risk Mitigation Report
# Objective: Quantify the capital freed up by transitioning to data-driven inventory management
# ==============================================================================

# Calculate Macro Financial Metrics
total_legacy_capital = optimized_df['Legacy_Capital_Tied_RM'].sum()
total_optimized_capital = optimized_df['Optimized_Capital_Tied_RM'].sum()
capital_saved = total_legacy_capital - total_optimized_capital
reduction_pct = (capital_saved / total_legacy_capital) * 100

# Executive Output
print("📊 MRO STRATEGIC IMPACT & CAPITAL ALLOCATION REPORT")
print("=" * 70)
print(f"🔴 Legacy Capital Tied Up : RM {total_legacy_capital:,.2f}")
print(f"🟢 Optimized Capital Tied Up: RM {total_optimized_capital:,.2f}")
print("-" * 70)
print(f"🚀 CAPITAL FREED UP       : RM {capital_saved:,.2f} ({reduction_pct:.2f}% Reduction)")
print("=" * 70)
print("✅ Outcome: Lean optimization drastically reduces excess capital while mathematically safeguarding against AOG (Aircraft on Ground) risks.")

# Top 5 Parts with Highest Capital Reduction
optimized_df['Capital_Savings_RM'] = optimized_df['Legacy_Capital_Tied_RM'] - optimized_df['Optimized_Capital_Tied_RM']
top_savings = optimized_df.sort_values(by='Capital_Savings_RM', ascending=False).head(5)

print("\\n🔍 TOP 5 HIGH-IMPACT COMPONENTS (Highest Capital Savings):")
display(top_savings[['Part_ID', 'Unit_Cost_RM', 'Legacy_Safety_Stock', 'Optimized_Safety_Stock', 'Capital_Savings_RM']])

📊 MRO STRATEGIC IMPACT & CAPITAL ALLOCATION REPORT
🔴 Legacy Capital Tied Up : RM 283,700,034.06
🟢 Optimized Capital Tied Up: RM 80,323,893.10
----------------------------------------------------------------------
🚀 CAPITAL FREED UP       : RM 203,376,140.96 (71.69% Reduction)
✅ Outcome: Lean optimization drastically reduces excess capital while mathematically safeguarding against AOG (Aircraft on Ground) risks.
\n🔍 TOP 5 HIGH-IMPACT COMPONENTS (Highest Capital Savings):


,Part_ID,Unit_Cost_RM,Legacy_Safety_Stock,Optimized_Safety_Stock,Capital_Savings_RM
30,HW-AV-1030,123041.62,135.0,12.0,15134119.26
13,HW-AV-1013,141529.85,135.0,33.0,14436044.70
28,HW-AV-1028,140157.67,123.0,32.0,12754347.97
48,HW-AV-1048,136058.53,99.0,14.0,11564975.05
20,HW-AV-1020,130834.48,99.0,11.0,11513434.24
